In [ ]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
from torchvision import models

from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


In [ ]:
input_csv_path = '/content/drive/MyDrive/pride_meme/PrideMM.csv'
images_folder_path = '/content/drive/MyDrive/pride_meme/Processed_images'
output_dir = '/content/drive/MyDrive/pride_meme'
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(input_csv_path)

# Keep only required columns
columns_to_drop = ['target', 'stance', 'humor', 'split']
df_processed = df.drop(columns=[c for c in columns_to_drop if c in df.columns])

df_processed = df_processed[['name', 'text', 'hate']]

# Remove duplicates & missing images
existing_images = set(os.listdir(images_folder_path))
df_processed = df_processed[df_processed['name'].isin(existing_images)]
df_processed = df_processed.drop_duplicates(subset=['name']).reset_index(drop=True)


In [ ]:
X = df_processed[['name', 'text']]
y = df_processed['hate']

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.1,
    stratify=y,
    random_state=42
)

train_df = pd.concat([X_train, y_train], axis=1).reset_index(drop=True)
val_df   = pd.concat([X_val, y_val], axis=1).reset_index(drop=True)

print("Train size:", len(train_df))
print("Val size:", len(val_df))


In [ ]:
MAX_TEXT_LENGTH = 128
BATCH_SIZE = 32//2
NUM_WORKERS = 2

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

image_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class PrideMMDataset(Dataset):
    def __init__(self, dataframe, img_dir, tokenizer, transform=None, max_length=128):
        self.df = dataframe
        self.img_dir = img_dir
        self.tokenizer = tokenizer
        self.transform = transform
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(os.path.join(self.img_dir, row['name'])).convert('RGB')
        if self.transform:
            image = self.transform(image)

        text = row['text'] if isinstance(row['text'], str) else ""
        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            "image": image,
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(row['hate'], dtype=torch.long)
        }

train_dataset = PrideMMDataset(train_df, images_folder_path, tokenizer, image_transforms)
val_dataset   = PrideMMDataset(val_df, images_folder_path, tokenizer, image_transforms)

train_dataloader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
)

val_dataloader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)


In [ ]:
class MultimodalModel(nn.Module):
    def __init__(self, image_encoder, text_encoder, proj_dim=256, num_classes=2):
        super().__init__()

        self.image_encoder = image_encoder
        self.text_encoder = text_encoder

        self.img_proj = nn.Linear(1024, proj_dim)
        self.txt_proj = nn.Linear(768, proj_dim)

        self.classifier = nn.Sequential(
            nn.Linear(proj_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, image, input_ids, attention_mask):
        img_feat = self.image_encoder(image)
        img_feat = img_feat.view(img_feat.size(0), -1) # Flatten the image features
        img_feat = self.img_proj(img_feat)

        txt_out = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        txt_feat = self.txt_proj(txt_out.last_hidden_state[:, 0])

        fused = torch.cat([img_feat, txt_feat], dim=1)
        return self.classifier(fused)  # RAW logits

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

convnext = models.convnext_base(weights="IMAGENET1K_V1")
convnext.classifier = nn.Identity()

roberta = AutoModel.from_pretrained("roberta-base")

# Freeze everything
for p in convnext.parameters():
    p.requires_grad = False
for p in roberta.parameters():
    p.requires_grad = False

# Unfreeze top layers
for p in convnext.features[2].parameters():
    p.requires_grad = True
for p in convnext.features[3].parameters():
    p.requires_grad = True

for i in [9, 10, 11]:
    for p in roberta.encoder.layer[i].parameters():
        p.requires_grad = True

model = MultimodalModel(convnext, roberta).to(device)


In [ ]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

criterion = nn.CrossEntropyLoss()


In [ ]:
def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return acc, prec, rec, f1


In [ ]:
# @torch.no_grad()
# def evaluate(model, dataloader):
#     model.eval()
#     y_true, y_pred = [], []

#     for batch in dataloader:
#         logits = model(
#             batch['image'].to(device),
#             batch['input_ids'].to(device),
#             batch['attention_mask'].to(device)
#         )
#         preds = torch.argmax(logits, dim=1)

#         y_true.extend(batch['labels'].numpy())
#         y_pred.extend(preds.cpu().numpy())

#     return compute_metrics(y_true, y_pred)


# EPOCHS = 10

# for epoch in range(EPOCHS):
#     model.train()
#     train_loss = 0
#     y_true, y_pred = [], []

#     for batch in train_dataloader:
#         optimizer.zero_grad()

#         logits = model(
#             batch['image'].to(device),
#             batch['input_ids'].to(device),
#             batch['attention_mask'].to(device)
#         )
#         loss = criterion(logits, batch['labels'].to(device))
#         loss.backward()
#         optimizer.step()

#         train_loss += loss.item()
#         y_true.extend(batch['labels'].numpy())
#         y_pred.extend(torch.argmax(logits, dim=1).cpu().numpy())

#     train_metrics = compute_metrics(y_true, y_pred)
#     val_metrics = evaluate(model, val_dataloader)

#     print(
#         f"\nEpoch {epoch+1}/{EPOCHS}"
#         f"\nTrain | Loss: {train_loss/len(train_dataloader):.4f} "
#         f"Acc: {train_metrics[0]:.4f} Prec: {train_metrics[1]:.4f} "
#         f"Rec: {train_metrics[2]:.4f} F1: {train_metrics[3]:.4f}"
#         f"\nVal   | Acc: {val_metrics[0]:.4f} Prec: {val_metrics[1]:.4f} "
#         f"Rec: {val_metrics[2]:.4f} F1: {val_metrics[3]:.4f}"
#     )


In [ ]:
@torch.no_grad()
def evaluate(model, dataloader):
    model.eval()
    y_true, y_pred = [], []

    for batch in dataloader:
        logits = model(
            batch['image'].to(device),
            batch['input_ids'].to(device),
            batch['attention_mask'].to(device)
        )
        preds = torch.argmax(logits, dim=1)

        y_true.extend(batch['labels'].numpy())
        y_pred.extend(preds.cpu().numpy())

    return compute_metrics(y_true, y_pred)


EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    y_true, y_pred = [], []

    for batch in train_dataloader:
        optimizer.zero_grad()

        logits = model(
            batch['image'].to(device),
            batch['input_ids'].to(device),
            batch['attention_mask'].to(device)
        )
        loss = criterion(logits, batch['labels'].to(device))
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        y_true.extend(batch['labels'].numpy())
        y_pred.extend(torch.argmax(logits, dim=1).cpu().numpy())

    train_metrics = compute_metrics(y_true, y_pred)
    val_metrics = evaluate(model, val_dataloader)

    print(
        f"\nEpoch {epoch+1}/{EPOCHS}"
        f"\nTrain | Loss: {train_loss/len(train_dataloader):.4f} "
        f"Acc: {train_metrics[0]:.4f} Prec: {train_metrics[1]:.4f} "
        f"Rec: {train_metrics[2]:.4f} F1: {train_metrics[3]:.4f}"
        f"\nVal   | Acc: {val_metrics[0]:.4f} Prec: {val_metrics[1]:.4f} "
        f"Rec: {val_metrics[2]:.4f} F1: {val_metrics[3]:.4f}"
    )